# Information Retrieval Coursework Pipeline — Official ColBERTv2 Setup

## 1. Install dependencies (official ColBERT route)

Run this in a **fresh Colab runtime** with **T4 GPU** enabled.

This version follows the cleaner **official ColBERT** setup pattern from the intro notebook, while keeping your BM25 / BM25F / RM3 pipeline unchanged.


In [1]:
# ===== OFFICIAL COLBERT SETUP FOR COLAB =====

import importlib.util
import os
import sys

# Java for Pyserini / Lucene
!apt-get update -qq
!apt-get install -y -qq openjdk-21-jdk-headless
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
!java -version

# Core IR packages
base_packages = ["ir_datasets", "pyserini", "spacy", "pandas", "tqdm"]
missing_base = [p for p in base_packages if importlib.util.find_spec(p) is None]
if missing_base:
    !pip install -q ir_datasets pyserini spacy pandas tqdm

# Clone official ColBERT if needed
!git -C ColBERT pull || git clone https://github.com/stanford-futuredata/ColBERT.git

# Install official ColBERT package (close to the intro notebook pattern)
try:
    import google.colab
    !pip install -U pip
    !pip install -e ColBERT/['faiss-gpu','torch']
except Exception:
    !pip install -e ColBERT/

# SpaCy English model
try:
    import en_core_web_sm
except Exception:
    !python -m spacy download en_core_web_sm -q

print("✅ Setup complete.")
print("If ColBERT or dependencies were installed above, restart the runtime once before continuing.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)
Already up to date.
Obtaining file:///content/ColBERT
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached bitarray-3.8.0-cp312-cp312-man

## 2. Mount Google Drive & verify index path

Your Lucene index should be at:
```
/content/drive/MyDrive/Data/indexes/trec_covid_lemmatized
```


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

INDEX_PATH = "/content/drive/MyDrive/Data/indexes/trec_covid_lemmatized"

print("Index path exists:      ", os.path.exists(INDEX_PATH))
print("Index path is directory:", os.path.isdir(INDEX_PATH))
!ls "{INDEX_PATH}" | head -10


Index path exists:       True
Index path is directory: True
_0.fdm
_0.fdt
_0.fdx
_0.fnm
_0_Lucene90_0.dvd
_0_Lucene90_0.dvm
_0_Lucene99_0.doc
_0_Lucene99_0.pos
_0_Lucene99_0.tim
_0_Lucene99_0.tip


## 3. Imports & dataset loading

In [4]:
import os, json, logging
from collections import Counter
from itertools import islice

import ir_datasets
import pandas as pd
import spacy
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher

# Suppress noisy Java / Pyserini warnings
logging.disable(logging.WARNING)
os.environ["JDK_JAVA_OPTIONS"] = "--add-modules jdk.incubator.vector"

dataset = ir_datasets.load("cord19/trec-covid/round1")
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

print("Dataset loaded.")
print("Document fields:", dataset.docs_cls()._fields)


Dataset loaded.
Document fields: ('doc_id', 'title', 'doi', 'date', 'abstract')


## 4. Optional: dataset inspection
Run these cells to understand the collection and qrels — not required for the pipeline.

In [5]:
# Show 3 example documents
for doc in islice(dataset.docs_iter(), 3):
    print(f"ID: {doc.doc_id}")
    print(f"Title: {doc.title}")
    print(f"Date: {doc.date}")
    print(f"Abstract: {(doc.abstract or '')[:400]}")
    print("-" * 80)


docs_iter: 0doc [00:00, ?doc/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-04-10/metadata.csv: 0.0%| 0.00/77.3M [00:00<?, ?B/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-04-10/metadata.csv: 8.5%| 6.59M/77.3M [00:00<00:01, 65.4MB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-04-10/metadata.csv: 18.7%| 14.5M/77.3M [00:00<00:00, 72.2MB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-04-10/metadata.csv: 28.9%| 22.4M/77.3M [00:00<00:00, 73.4MB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-04-10/metadata.csv: 39.2%| 30.3M/77.3M [00:00<00:00, 70.5MB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-04-10/metadata.csv: 49.4%| 38.2M/77.3M [00:00<00:00, 62.2MB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-04-10/metadata.csv: 59.6%| 46.1M/77.3M [00:00<00:00, 63.3MB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com

ID: xqhn0vbp
Title: Airborne rhinovirus detection and effect of ultraviolet irradiation on detection by a semi-nested RT-PCR assay
Date: 2003-01-13
Abstract: BACKGROUND: Rhinovirus, the most common cause of upper respiratory tract infections, has been implicated in asthma exacerbations and possibly asthma deaths. Although the method of transmission of rhinoviruses is disputed, several studies have demonstrated that aerosol transmission is a likely method of transmission among adults. As a first step in studies of possible airborne rhinovirus transmissi
--------------------------------------------------------------------------------
ID: gi6uaa83
Title: Discovering human history from stomach bacteria
Date: 2003-04-28
Abstract: Recent analyses of human pathogens have revealed that their evolutionary histories are congruent with the hypothesized pattern of ancient and modern human population migrations. Phylogenetic trees of strains of the bacterium Helicobacter pylori and the polyoma JC v

In [6]:
# Count judged documents per query
qrel_counts = Counter(q.query_id for q in dataset.qrels_iter())
qrel_df = pd.DataFrame([
    {"query_id": qid, "num_judgments": count}
    for qid, count in sorted(qrel_counts.items(), key=lambda x: int(x[0]))
])
qrel_df.head(10)


,query_id,num_judgments
0,1,323
1,2,284
2,3,337
3,4,357
4,5,336
5,6,321
6,7,275
7,8,360
8,9,298
9,10,191


## 5. Query preprocessing & searcher setup

- **Person 1**: BM25 and BM25F lexical retrieval
- **Person 2**: Preprocessing + RM3 query expansion
- **Person 3**: ColBERTv2 reranking from Run C candidates


In [7]:
def preprocess_query(query: str) -> str:
    """Lowercase + stop-word removal + lemmatisation."""
    return " ".join(
        t.lemma_.lower()
        for t in nlp(query)
        if not t.is_stop and not t.is_punct and not t.is_space
    )

# Three separate searchers so each run is independently reproducible
bm25_searcher = LuceneSearcher(INDEX_PATH)
bm25_searcher.set_bm25(k1=0.9, b=0.4)

bm25f_searcher = LuceneSearcher(INDEX_PATH)
bm25f_searcher.set_bm25(k1=1.2, b=0.75)

rm3_searcher = LuceneSearcher(INDEX_PATH)
rm3_searcher.set_bm25(k1=1.2, b=0.75)
rm3_searcher.set_rm3(fb_terms=10, fb_docs=10, original_query_weight=0.5)

print("BM25, BM25F, and RM3 searchers ready.")


BM25, BM25F, and RM3 searchers ready.


In [8]:
def build_topic_text(topic) -> str:
    """Concatenate title + description/question, then preprocess."""
    title    = getattr(topic, "title", "") or ""
    question = getattr(topic, "description", "") or getattr(topic, "question", "") or ""
    return preprocess_query(f"{title} {question}".strip())

def search_trec_covid_bm25(query: str, k: int = 100):
    return bm25_searcher.search(preprocess_query(query), k=k)

def search_trec_covid_bm25f(query: str, k: int = 100,
                             title_weight: float = 2.0, abstract_weight: float = 1.0):
    return bm25f_searcher.search(
        preprocess_query(query), k=k,
        fields={"title": title_weight, "abstract": abstract_weight}
    )

def search_trec_covid_rm3(topic_text: str, k: int = 100,
                           title_weight: float = 2.0, abstract_weight: float = 1.0):
    return rm3_searcher.search(
        topic_text, k=k,
        fields={"title": title_weight, "abstract": abstract_weight}
    )

def display_hits(hits, searcher, max_display: int = 5):
    for rank, hit in enumerate(hits[:max_display], start=1):
        doc_data = json.loads(searcher.doc(hit.docid).raw())
        print(f"Rank:  {rank}")
        print(f"ID:    {hit.docid}")
        print(f"Score: {hit.score:.4f}")
        print(f"Title: {doc_data.get('title', 'No Title Found')}")
        print("-" * 60)


## 6. Single-query sanity checks

In [9]:
raw_query = "covid-19 in kids"

print("=== BM25 Results ===")
display_hits(search_trec_covid_bm25(raw_query, k=5), bm25_searcher)


=== BM25 Results ===
Rank:  1
ID:    wy9k0508
Score: 9.0452
Title: covid 19 pandemic gynaecological laparoscopic surgery known unknown
------------------------------------------------------------
Rank:  2
ID:    0h28a3az
Score: 8.9379
Title: prophylaxis paromomycin natural cryptosporidial infection neonatal kid
------------------------------------------------------------
Rank:  3
ID:    4707pec6
Score: 8.9036
Title: frequency testing covid 19 infection presence high number available bed country predict outcome infection gdp country descriptive statistical analysis
------------------------------------------------------------
Rank:  4
ID:    20o4ufa3
Score: 8.7883
Title: rotavirus excretion kid naturally infect goat herd
------------------------------------------------------------
Rank:  5
ID:    nnlynjoy
Score: 8.2446
Title: effect continuous renal replacement therapy cause mortality covid-19 patient undergo invasive mechanical ventilation retrospective cohort study
--------------------

In [10]:
print("=== BM25F Results ===")
display_hits(
    search_trec_covid_bm25f(raw_query, k=5, title_weight=10.0, abstract_weight=1.0),
    bm25f_searcher
)


=== BM25F Results ===
Rank:  1
ID:    ylr2b8ck
Score: 78.3374
Title: covid 19 intra cerebral hemorrhage causative coincidental
------------------------------------------------------------
Rank:  2
ID:    wy9k0508
Score: 76.3556
Title: covid 19 pandemic gynaecological laparoscopic surgery known unknown
------------------------------------------------------------
Rank:  3
ID:    6ytxwx94
Score: 74.6326
Title: effect covid 19 pandemic daily life
------------------------------------------------------------
Rank:  4
ID:    gzi4tjjv
Score: 74.6326
Title: case neonatal infection covid 19 spain
------------------------------------------------------------
Rank:  5
ID:    j00m2ctc
Score: 73.6679
Title: anesthetic management patients covid 19 infections emergency procedures
------------------------------------------------------------


## 7. Load all TREC-COVID topics

In [11]:
query_records = list(dataset.queries_iter())

queries = {
    q.query_id: {
        "title":       getattr(q, "title", "")       or "",
        "description": getattr(q, "description", "") or getattr(q, "question", "") or "",
        "narrative":   getattr(q, "narrative", "")   or "",
    }
    for q in query_records
}

query_texts = {
    qid: build_topic_text(type("TopicObj", (), fields)())
    for qid, fields in queries.items()
}

first_qid = list(query_texts.keys())[0]
print(f"Number of queries: {len(query_texts)}")
print(f"Example query ID:  {first_qid}")
print(f"Title:             {queries[first_qid]['title']}")
print(f"Description:       {queries[first_qid]['description']}")
print(f"Search text:       {query_texts[first_qid]}")


Number of queries: 30
Example query ID:  1
Title:             coronavirus origin
Description:       what is the origin of COVID-19
Search text:       coronavirus origin origin covid-19


## 8. Run BM25 / BM25F / RM3 for all queries → export TREC runs

In [12]:
def save_trec_run(results_dict, output_file, run_name):
    with open(output_file, "w") as f:
        for qid, docs in results_dict.items():
            for rank, (doc_id, score) in enumerate(docs, start=1):
                f.write(f"{qid} Q0 {doc_id} {rank} {score:.6f} {run_name}\n")
    print(f"Saved → {output_file}")

def run_all_queries_bm25(query_texts, top_k=100):
    results = {}
    for qid, text in tqdm(query_texts.items(), desc="BM25"):
        results[qid] = [(h.docid, h.score) for h in search_trec_covid_bm25(text, k=top_k)]
    return results

def run_all_queries_bm25f(query_texts, top_k=100, title_weight=10.0, abstract_weight=1.0):
    results = {}
    for qid, text in tqdm(query_texts.items(), desc="BM25F"):
        results[qid] = [(h.docid, h.score) for h in search_trec_covid_bm25f(
            text, k=top_k, title_weight=title_weight, abstract_weight=abstract_weight)]
    return results

def run_all_queries_rm3(query_texts, top_k=100, title_weight=10.0, abstract_weight=1.0):
    results = {}
    for qid, text in tqdm(query_texts.items(), desc="BM25F+RM3"):
        results[qid] = [(h.docid, h.score) for h in search_trec_covid_rm3(
            text, k=top_k, title_weight=title_weight, abstract_weight=abstract_weight)]
    return results


In [13]:
# Run A: BM25
bm25_all_results = run_all_queries_bm25(query_texts, top_k=100)
save_trec_run(bm25_all_results, "runA_bm25.txt", "runA_bm25")

# Run B: BM25F
bm25f_all_results = run_all_queries_bm25f(query_texts, top_k=100,
                                           title_weight=10.0, abstract_weight=1.0)
save_trec_run(bm25f_all_results, "runB_bm25f.txt", "runB_bm25f")

# Run C: BM25F + RM3  (used as ColBERT candidate pool)
rm3_all_results = run_all_queries_rm3(query_texts, top_k=100,
                                       title_weight=10.0, abstract_weight=1.0)
save_trec_run(rm3_all_results, "runC_bm25f_rm3.txt", "runC_bm25f_rm3")


BM25: 100%|██████████| 30/30 [00:01<00:00, 17.59it/s]


Saved → runA_bm25.txt


BM25F: 100%|██████████| 30/30 [00:00<00:00, 58.56it/s]


Saved → runB_bm25f.txt


BM25F+RM3: 100%|██████████| 30/30 [00:08<00:00,  3.66it/s]

Saved → runC_bm25f_rm3.txt


## 9. Build candidate documents for reranking

Uses **Run C** (BM25F + RM3) as the candidate pool for ColBERTv2, as specified in the assignment.


In [14]:
def get_doc_text(searcher, doc_id: str) -> str:
    doc = searcher.doc(doc_id)
    if doc is None:
        return ""
    raw = json.loads(doc.raw())
    title    = raw.get("title", "")    or ""
    abstract = raw.get("abstract", "") or ""
    return (title + " " + abstract).strip()

def build_candidate_docs(results_dict, max_docs_per_query: int = 100):
    candidate_docs = {}
    for qid, docs in tqdm(results_dict.items(), desc="Building candidate docs"):
        candidate_docs[qid] = [
            {"doc_id": doc_id, "text": get_doc_text(bm25f_searcher, doc_id), "score": score}
            for doc_id, score in docs[:max_docs_per_query]
        ]
    return candidate_docs

candidate_docs = build_candidate_docs(rm3_all_results, max_docs_per_query=100)
first_qid = list(candidate_docs.keys())[0]
print(f"Candidates for query '{first_qid}': {len(candidate_docs[first_qid])} docs")
candidate_docs[first_qid][:2]


Building candidate docs: 100%|██████████| 30/30 [00:00<00:00, 45.52it/s]

Candidates for query '1': 100 docs


[{'doc_id': 'gzi4tjjv',
  'text': 'case neonatal infection covid 19 spain',
  'score': 1.8555999994277954},
 {'doc_id': 'wy9k0508',
  'text': 'covid 19 pandemic gynaecological laparoscopic surgery known unknown worldwide impact covid 19 continue feel hospital country reduce elective non urgent case allow staffing resource deploy urgent gynaecological cancer procedure continue imperative theatre staff protect risk sars cov-2 viral transmission reduce operate asymptomatic suspect confirm covid 19 patient particular concern relate transmission covid 19 gynaecological laparoscopic surgery arise potential generation sars cov-2 contaminate aerosol co(2 leakage creation smoke use energy device aim paper review date evidence include experience china italy guide safe management patient undergo gynaecological procedure',
  'score': 1.7508000135421753}]

## 10. Official ColBERTv2 reranking (Person 3)

This notebook now uses the **official ColBERT** package instead of RAGatouille.

The plan is:

1. take the **Run C** candidate documents
2. build one temporary ColBERT collection from the union of those candidates
3. index that smaller collection
4. search it with ColBERTv2
5. filter results back to each query's own candidate set
6. save **Run D**

This stays close to the Colab intro notebook while still matching your assignment pipeline.


### Fast-start notes

- Use **Runtime → Change runtime type → T4 GPU**
- The first ColBERT load and indexing step can take a few minutes
- Start with a smaller retrieval depth such as **top-30**
- Only scale up after the sanity check works


In [15]:
# Verify GPU and import official ColBERT

!pip install ujson # Install missing ujson package

import sys
import time
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU is not enabled. In Colab go to Runtime → Change runtime type → T4 GPU."
    )

sys.path.insert(0, "ColBERT/")

from colbert import Indexer, Searcher
from colbert.infra import Run, RunConfig, ColBERTConfig

print("✅ Official ColBERT imported successfully.")

  Using cached ujson-5.12.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (9.6 kB)
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
✅ Official ColBERT imported successfully.


In [16]:
# Build one temporary ColBERT collection from the union of Run C candidates

from collections import OrderedDict
from pathlib import Path
import json

def build_colbert_collection(candidate_docs, max_docs_per_query=30, out_dir="colbert_tmp"):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    unique_docs = OrderedDict()
    qid_to_allowed = {}

    for qid, docs in candidate_docs.items():
        subset = docs[:max_docs_per_query]
        qid_to_allowed[qid] = set()

        for d in subset:
            doc_id = d["doc_id"]
            text = (d.get("text") or "").strip()

            if not text:
                continue

            qid_to_allowed[qid].add(doc_id)
            if doc_id not in unique_docs:
                unique_docs[doc_id] = text

    doc_ids = list(unique_docs.keys())
    collection = [unique_docs[doc_id] for doc_id in doc_ids]
    pid_to_docid = {pid: doc_id for pid, doc_id in enumerate(doc_ids)}

    with open(f"{out_dir}/pid_to_docid.json", "w") as f:
        json.dump(pid_to_docid, f)

    with open(f"{out_dir}/collection.tsv", "w") as f:
        for pid, text in enumerate(collection):
            clean_text = text.replace("\n", " ").replace("\t", " ")
            f.write(f"{pid}\t{clean_text}\n")

    return collection, pid_to_docid, qid_to_allowed

collection_small, pid_to_docid, qid_to_allowed = build_colbert_collection(
    candidate_docs,
    max_docs_per_query=30,
    out_dir="colbert_tmp"
)

print("Unique candidate docs in temporary ColBERT collection:", len(collection_small))
print("Example temporary collection entry:")
print(collection_small[0][:300] if collection_small else "No documents found")


Unique candidate docs in temporary ColBERT collection: 602
Example temporary collection entry:
case neonatal infection covid 19 spain


In [17]:
import shutil
import os

index_path = "ir_colbert/indexes/trec_covid_runC_candidates_top30"

if os.path.exists(index_path):
    print("Deleting old ColBERT index...")
    shutil.rmtree(index_path)

print("Ready to rebuild index.")

Ready to rebuild index.


In [ ]:
# Index the temporary candidate collection with official ColBERTv2

checkpoint = "colbert-ir/colbertv2.0"
index_name = "trec_covid_runC_candidates_top30"

start = time.time()

with Run().context(RunConfig(nranks=1, experiment="cw_ir_colbert")):
    config = ColBERTConfig(
        doc_maxlen=180,
        nbits=2,
        kmeans_niters=4
    )

    indexer = Indexer(checkpoint=checkpoint, config=config)
    indexer.index(
        name=index_name,
        collection=collection_small,
        overwrite=True
    )

print(f"✅ Temporary ColBERT index built in {time.time() - start:.1f} seconds.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


artifact.metadata: 0.00B [00:00, ?B/s]



[Apr 02, 09:42:31] #> Creating directory /content/experiments/cw_ir_colbert/indexes/trec_covid_runC_candidates_top30 


#> Starting...


In [ ]:
# Create the ColBERT searcher

start = time.time()

with Run().context(RunConfig(experiment="cw_ir_colbert")):
    searcher = Searcher(index=index_name, collection=collection_small)

print(f"✅ ColBERT searcher ready in {time.time() - start:.1f} seconds.")


In [ ]:
# Rerank each query against its Run C candidate set using official ColBERT search

from tqdm import tqdm

def rerank_with_official_colbert(query_texts, qid_to_allowed, pid_to_docid, searcher, search_k=200):
    final_results = {}

    for qid, query_text in tqdm(query_texts.items(), desc="Official ColBERTv2 reranking"):
        allowed_docids = qid_to_allowed.get(qid, set())

        if not allowed_docids:
            final_results[qid] = []
            continue

        pids, ranks, scores = searcher.search(query_text, k=search_k)

        ranked_docs = []
        seen = set()

        for pid, rank, score in zip(pids, ranks, scores):
            doc_id = pid_to_docid.get(int(pid))
            if doc_id in allowed_docids and doc_id not in seen:
                ranked_docs.append((doc_id, float(score)))
                seen.add(doc_id)

            if len(ranked_docs) >= len(allowed_docids):
                break

        final_results[qid] = ranked_docs

    return final_results


In [ ]:
# 1-query sanity check

sample_qids = list(query_texts.keys())[:1]

colbert_sample_results = rerank_with_official_colbert(
    {qid: query_texts[qid] for qid in sample_qids},
    qid_to_allowed=qid_to_allowed,
    pid_to_docid=pid_to_docid,
    searcher=searcher,
    search_k=100
)

print("Sample result:")
print(colbert_sample_results[sample_qids[0]][:5])


## 11. Save the reranked run (Run D)

Recommended order:

1. 1 query sanity check
2. 3-query test
3. all queries


In [ ]:
def save_trec_reranked_run(results, output_file="runD_colbertv2.txt", run_name="runD_colbertv2"):
    with open(output_file, "w") as f:
        for qid, docs in results.items():
            for rank, (doc_id, score) in enumerate(docs, start=1):
                f.write(f"{qid} Q0 {doc_id} {rank} {score:.6f} {run_name}\n")
    print(f"Saved → {output_file}")

# 3-query test
sample_qids = list(query_texts.keys())[:3]
colbert_3q_results = rerank_with_official_colbert(
    {qid: query_texts[qid] for qid in sample_qids},
    qid_to_allowed=qid_to_allowed,
    pid_to_docid=pid_to_docid,
    searcher=searcher,
    search_k=100
)

for qid in sample_qids:
    print(qid, colbert_3q_results[qid][:3])

# Full Run D (uncomment once the sanity checks work)
# colbert_all_results = rerank_with_official_colbert(
#     query_texts,
#     qid_to_allowed=qid_to_allowed,
#     pid_to_docid=pid_to_docid,
#     searcher=searcher,
#     search_k=100
# )
# save_trec_reranked_run(colbert_all_results, "runD_colbertv2.txt", "runD_colbertv2")


## 12. Run summary & parameters

| Run file | Method | Notes |
|---|---|---|
| `runA_bm25.txt` | BM25 | lexical baseline |
| `runB_bm25f.txt` | BM25F | field-aware lexical retrieval |
| `runC_bm25f_rm3.txt` | BM25F + RM3 | candidate set for neural stage |
| `runD_colbertv2.txt` | ColBERTv2 over Run C candidates | official ColBERT reranking |

### Suggested settings to report
- BM25: `k1=0.9`, `b=0.4`
- BM25F: title weight `10.0`, abstract weight `1.0`
- RM3: `fb_terms=10`, `fb_docs=10`, `original_query_weight=0.5`
- ColBERTv2 reranking depth: start with top-30 candidates, then scale up if runtime allows
- Text unit reranked: `title + abstract`
